# Data Modeling

## Objective

The objective of this notebook is to analyze the relationships between the Olist datasets and design a dimensional model optimized for analytical purposes.

The modeling process aims to identify:

- Fact tables
- Dimension tables
- Primary keys
- Foreign keys
- Relationships between entities

## Dataset Loading

The datasets required for the analytical model are loaded in this notebook to validate relationships, keys, and cardinality between entities.

The main entities analyzed are:

- Customers
- Orders
- Order Items
- Products
- Sellers
- Payments
- Reviews

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

In [3]:
customers = pd.read_csv(RAW_DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DATA_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DATA_PATH / "olist_sellers_dataset.csv")
payments = pd.read_csv(RAW_DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv")

In [4]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews
}

overview = []

for name, df in tables.items():
    overview.append({
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

pd.DataFrame(overview)

,Table,Rows,Columns
0,customers,99441,5
1,orders,99441,8
2,order_items,112650,7
3,products,32951,9
4,sellers,3095,4
5,payments,103886,5
6,reviews,99224,7


## Relationship Validation

### Orders → Customers

The `orders` table contains customer information through the `customer_id` field.

Expected relationship:

`customers.customer_id` (Primary Key) → `orders.customer_id` (Foreign Key)

Cardinality:

One customer can have multiple orders (1:N).

In [10]:
orders["customer_id"].isin(customers["customer_id"]).value_counts()

customer_id
True    99441
Name: count, dtype: int64

In [11]:
orders.groupby("customer_id")["order_id"].count().describe()

count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: order_id, dtype: float64

### Order Items → Orders

The `order_items` table contains product-level information for each order.

Expected relationship:

`orders.order_id` (Primary Key) → `order_items.order_id` (Foreign Key)

Cardinality:

One order can contain multiple items (1:N).

In [12]:
order_items["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    112650
Name: count, dtype: int64

### Order Items → Products

Each item references a product through `product_id`.

Expected relationship:

`products.product_id` → `order_items.product_id`

Cardinality:

One product can appear in multiple orders (1:N).

In [13]:
order_items["product_id"].isin(products["product_id"]).value_counts()

product_id
True    112650
Name: count, dtype: int64

### Order Items → Sellers

Each item is associated with a seller.

Expected relationship:

`sellers.seller_id` → `order_items.seller_id`

Cardinality:

One seller can sell multiple products (1:N).

In [14]:
order_items["seller_id"].isin(sellers["seller_id"]).value_counts()

seller_id
True    112650
Name: count, dtype: int64

### Customer → Orders Relationship

The validation confirms that all orders have a corresponding customer record.

Although the expected relationship is one-to-many (1:N), the dataset contains one order per customer_id. This happens because Olist assigns a unique customer_id for each order, while customer_unique_id represents the actual customer identity across multiple purchases.

Therefore:

- customer_id acts as the transactional customer reference.
- customer_unique_id should be considered when analyzing customer lifetime behavior.

### Orders → Order Items Relationship

All order items have a valid corresponding order.

The relationship follows a one-to-many structure:

orders.order_id → order_items.order_id

One order may contain multiple items.

### Products → Order Items Relationship

All order items reference existing products.

The relationship follows:

products.product_id → order_items.product_id

A product can appear in multiple orders.

### Sellers → Order Items Relationship

All order items reference existing sellers.

The relationship follows:

sellers.seller_id → order_items.seller_id

One seller can be associated with multiple order items.

## Relationship Summary

| Relationship | Cardinality | Description |
|---|---|---|
| customers → orders | 1:N | Customers are linked to orders through customer_id |
| orders → order_items | 1:N | Orders may contain multiple items |
| products → order_items | 1:N | Products can appear in multiple orders |
| sellers → order_items | 1:N | Sellers can have multiple sold items |

All tested foreign key relationships were validated successfully, with no orphan records detected.

In [15]:
payments["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    103886
Name: count, dtype: int64

In [16]:
reviews["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    99224
Name: count, dtype: int64

### Orders → Payments Relationship

The `payments` table contains payment information associated with each order.

Expected relationship:

`orders.order_id` → `payments.order_id`

Cardinality:

One order can have multiple payment records (1:N).

All payment records were successfully matched with existing orders, with no orphan records detected.

### Orders → Reviews Relationship

The `reviews` table stores customer feedback associated with orders.

Expected relationship:

`orders.order_id` → `reviews.order_id`

Cardinality:

One order may have one review (1:1).

Not every order contains a review, since customer feedback is optional.

All available reviews were successfully matched with existing orders.

# Final Relationship Overview

After validating primary keys and foreign keys, the following relationships were confirmed:

| Parent Table | Child Table | Key | Cardinality |
|---|---|---|---|
| customers | orders | customer_id | 1:N |
| orders | order_items | order_id | 1:N |
| products | order_items | product_id | 1:N |
| sellers | order_items | seller_id | 1:N |
| orders | payments | order_id | 1:N |
| orders | reviews | order_id | 1:1 |

No orphan records were identified during relationship validation.

# Star Schema Design

## Objective

Based on the validated relationships, a dimensional model was designed to support analytical queries and business intelligence reporting.

The model separates transactional data into:

- Fact tables: containing measurable business events.
- Dimension tables: containing descriptive attributes used for analysis.

The main analytical grain selected is the order item level, allowing detailed analysis of sales performance, products, sellers, and customers.

## Fact Table

### fact_sales

**Grain:**

One row per purchased product item.

The `order_items` table was selected as the main source because it represents the lowest level of sales detail available in the dataset.

### Measures:

- Product price
- Freight value

### Keys:

- order_id
- order_item_id
- customer_id
- product_id
- seller_id

| Column | Description |
|---|---|
| order_id | Identifier of the order |
| order_item_id | Sequential item number within an order |
| customer_id | Customer reference |
| product_id | Product reference |
| seller_id | Seller reference |
| price | Product selling price |
| freight_value | Shipping cost |

## dim_customer

Contains customer attributes used for customer segmentation and geographic analysis.

Source table:

customers


| Column | Description |
|---|---|
| customer_id | Transactional customer identifier |
| customer_unique_id | Customer identifier across multiple purchases |
| customer_city | Customer city |
| customer_state | Customer state |
| customer_zip_code_prefix | ZIP code prefix |

## dim_product

Contains product characteristics used for product analysis.

Source table:

products


| Column | Description |
|---|---|
| product_id | Product identifier |
| product_category_name | Product category |
| product_weight_g | Product weight |
| product_length_cm | Product length |
| product_height_cm | Product height |
| product_width_cm | Product width |

## dim_seller

Contains seller information.

Source table:

sellers


| Column | Description |
|---|---|
| seller_id | Seller identifier |
| seller_city | Seller city |
| seller_state | Seller state |
| seller_zip_code_prefix | ZIP code prefix |

## dim_date

A dedicated date dimension will be created during the ETL phase to support time-based analysis.

Expected attributes:

- Date
- Year
- Month
- Quarter
- Day

## fact_payment

Contains payment transactions associated with orders.

Source table:

order_payments


Grain:

One row per payment transaction.


Measures:

- payment_value


Attributes:

- payment_type
- payment_installments

## fact_review

Contains customer satisfaction information.

Source table:

order_reviews


Grain:

One row per customer review.


Measures:

- review_score


Attributes:

- review_creation_date
- review_answer_timestamp

# Final Data Model

The final analytical model follows a dimensional modeling approach using a fact constellation schema.

```mermaid
flowchart TD

    dim_customer[dim_customer]
    dim_order[dim_order]

    dim_product[dim_product]
    dim_seller[dim_seller]
    dim_date[dim_date]

    fact_sales{{fact_sales}}
    fact_payments{{fact_payments}}
    fact_review{{fact_review}}


    dim_customer --> dim_order

    dim_order --> fact_sales
    dim_order --> fact_payments
    dim_order --> fact_review

    dim_product --> fact_sales
    dim_seller --> fact_sales
    dim_date --> fact_sales
```

### Modeling Considerations

The model was designed considering the grain of each dataset:

- fact_sales: one row per order item.
- fact_payments: one row per payment transaction.
- fact_reviews: one row per customer review.

Shared entities such as customers and orders allow consistent analysis across different business processes.